<a href="https://colab.research.google.com/github/shravan1808/ML_SERIES/blob/Main/12_API_Driven_Decision_Tree_Classifier_Feature_Importance/notebook/Project_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [76]:
import requests
import io
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [77]:
def fetch_telemetry_data(url):
  try:
    response = requests.get(url)
    response.raise_for_status()
    data = pd.read_csv(io.StringIO(response.text))
    return data
  except requests.RequestException as e:
    print(f"Error fetching data: {e}")

In [78]:
HOST_TELEMETRY_API_ENDPOINT = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"

In [79]:
telemetry_df = fetch_telemetry_data(HOST_TELEMETRY_API_ENDPOINT)

In [80]:
print(telemetry_df.head())

   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa


In [81]:
telemetry_df.rename(columns={
    'sepal_length' : 'cpu_usage_pct',
    'sepal_width' : 'memory_usage_pct',
    'petal_length' : 'disk_io_rate',
    'petal_width' : 'network_latency_ms'
},inplace=True)

telemetry_df['Is_Failure'] = np.where(telemetry_df['species'] == 'setosa', 0, 1)

telemetry_df.drop(columns=['species'],inplace=True)

In [82]:
print(telemetry_df.head())

   cpu_usage_pct  memory_usage_pct  disk_io_rate  network_latency_ms  \
0            5.1               3.5           1.4                 0.2   
1            4.9               3.0           1.4                 0.2   
2            4.7               3.2           1.3                 0.2   
3            4.6               3.1           1.5                 0.2   
4            5.0               3.6           1.4                 0.2   

   Is_Failure  
0           0  
1           0  
2           0  
3           0  
4           0  


In [83]:
X = telemetry_df.drop(columns=['Is_Failure'],axis=1)
y = telemetry_df['Is_Failure']

In [84]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [85]:
X_train_size = X_train.shape[0]
X_test_size = X_test.shape[0]
y_train_counts = y_train.value_counts()
y_test_counts = y_test.value_counts()

In [86]:
model = DecisionTreeClassifier(criterion='gini',random_state=42)
model.fit(X_train,y_train)
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

In [87]:
train_accuracy = accuracy_score(y_train,train_pred)
test_accuracy = accuracy_score(y_test,test_pred)
accuracy_gap = train_accuracy - test_accuracy

In [88]:
print(train_accuracy*100,test_accuracy*100)
print(accuracy_gap*100)

100.0 100.0
0.0


In [89]:
print((y_train==train_pred).value_counts())

Is_Failure
True    120
Name: count, dtype: int64


In [90]:
print((y_test==test_pred).value_counts())

Is_Failure
True    30
Name: count, dtype: int64


In [91]:
feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)
for col in X.columns:
  print(f"{col:<15} : {feature_importance[X.columns.get_loc(col)]:.4f}")

cpu_usage_pct   : 1.0000
memory_usage_pct : 0.0000
disk_io_rate    : 0.0000
network_latency_ms : 0.0000


/tmp/ipykernel_5060/2212757710.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"{col:<15} : {feature_importance[X.columns.get_loc(col)]:.4f}")


In [92]:
rules = export_text(
    model,
    feature_names=list(X.columns)
)
print(rules)

|--- disk_io_rate <= 2.45
|   |--- class: 0
|--- disk_io_rate >  2.45
|   |--- class: 1



In [93]:
pruned_model = DecisionTreeClassifier(
    criterion='gini',
    random_state=42,
    max_depth=2,
    min_samples_leaf=5
)
pruned_model.fit(X_train,y_train)
train_pred_pruned = pruned_model.predict(X_train)
test_pred_pruned = pruned_model.predict(X_test)

In [94]:
pruned_train_accuracy = accuracy_score(y_train,train_pred_pruned)
pruned_test_accuracy = accuracy_score(y_test,test_pred_pruned)
pruned_accuracy_gap = pruned_train_accuracy - pruned_test_accuracy

In [95]:
print(pruned_train_accuracy*100,pruned_test_accuracy*100)
print(pruned_accuracy_gap*100)

100.0 100.0
0.0


In [98]:
print("\n========== API-DRIVEN DECISION TREE & OVERFITTING ENGINE ==========")

print("\nData Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)")
print(f"Master Dataset Records     : {telemetry_df.shape[0]} Telemetry Logs")
print("Features Included          : 4 Continuous Metrics (cpu_usage_pct, memory_usage_pct, disk_io_rate, network_latency_ms)")
print("Target Output              : Is_Failure (Binary Classification: 0 = Normal, 1 = Failure)")

print("\nModel Training Metrics:")
print(
    f"- Stratified Train Split   : {X_train_size} Records "
    f"({y_train_counts[0]} Normal / {y_train_counts[1]} Failure)"
)
print(
    f"- Stratified Test Split    : {X_test_size} Records "
    f"({y_test_counts[0]} Normal / {y_test_counts[1]} Failure)"
)
print(
    f"- Unpruned Tree Accuracy   : "
    f"Train = {train_accuracy * 100:.2f}% | "
    f"Test = {test_accuracy * 100:.2f}%"
)

print("\nFeature Importance Profiling:")

primary_feature = feature_importance.index[0]
secondary_feature = feature_importance.index[1]

print(
    f"- Primary Root Split Feature: "
    f"{primary_feature} "
    f"(Gini Importance = {feature_importance.iloc[0]:.4f})"
)

print(
    f"- Secondary Split Feature   : "
    f"{secondary_feature} "
    f"(Gini Importance = {feature_importance.iloc[1]:.4f})"
)

# print("\nExtracted Tree Decision Structure:")
# print(export_text(model, feature_names=list(X.columns)))

print("\nHyperparameter Pruning & Regularization:")
print(
    f"- Baseline Unpruned Tree    : "
    f"{test_accuracy * 100:.2f}% Test Accuracy "
    f"(Train/Test Gap = {accuracy_gap * 100:.2f}%)"
)

print(
    f"- Pruned Tree (max_depth=2) : "
    f"{pruned_test_accuracy * 100:.2f}% Test Accuracy "
    f"(Train/Test Gap = {pruned_accuracy_gap * 100:.2f}%)"
)

print("\nConclusion:")
print(
    "By fetching telemetry data dynamically over HTTP REST endpoints, "
    "the pipeline evaluates system health in real time. "
    "Decision Trees provide interpretable decision rules, while "
    "max_depth and min_samples_leaf control model complexity and "
    "help reduce overfitting."
)


========== API-DRIVEN DECISION TREE & OVERFITTING ENGINE ==========

Data Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)
Master Dataset Records     : 150 Telemetry Logs
Features Included          : 4 Continuous Metrics (cpu_usage_pct, memory_usage_pct, disk_io_rate, network_latency_ms)
Target Output              : Is_Failure (Binary Classification: 0 = Normal, 1 = Failure)

Model Training Metrics:
- Stratified Train Split   : 120 Records (40 Normal / 80 Failure)
- Stratified Test Split    : 30 Records (10 Normal / 20 Failure)
- Unpruned Tree Accuracy   : Train = 100.00% | Test = 100.00%

Feature Importance Profiling:
- Primary Root Split Feature: disk_io_rate (Gini Importance = 1.0000)
- Secondary Split Feature   : cpu_usage_pct (Gini Importance = 0.0000)

Hyperparameter Pruning & Regularization:
- Baseline Unpruned Tree    : 100.00% Test Accuracy (Train/Test Gap = 0.00%)
- Pruned Tree (max_depth=2) : 100.00% Test Accuracy (Train/Test Gap = 0.00%)

Conclusion:
By 